In [1]:
import pandas as pd  
import sklearn as sk 
import matplotlib.pyplot as plt 

In [2]:
!kaggle competitions download -c playground-series-s5e8 -q 

In [3]:
!unzip -qo playground-series-s5e8.zip 
!rm playground-series-s5e8.zip

In [4]:
train_df = pd.read_csv("train.csv") 
test_df = pd.read_csv("test.csv")
submission_df = pd.read_csv("sample_submission.csv") 

In [5]:
train_df.head()

,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,0,42,technician,married,secondary,no,7,no,no,cellular,25,aug,117,3,-1,0,unknown,0
1,1,38,blue-collar,married,secondary,no,514,no,no,unknown,18,jun,185,1,-1,0,unknown,0
2,2,36,blue-collar,married,secondary,no,602,yes,no,unknown,14,may,111,2,-1,0,unknown,0
3,3,27,student,single,secondary,no,34,yes,no,unknown,28,may,10,2,-1,0,unknown,0
4,4,26,technician,married,secondary,no,889,yes,no,cellular,3,feb,902,1,-1,0,unknown,1


In [6]:
test_df.head()

,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome
0,750000,32,blue-collar,married,secondary,no,1397,yes,no,unknown,21,may,224,1,-1,0,unknown
1,750001,44,management,married,tertiary,no,23,yes,no,cellular,3,apr,586,2,-1,0,unknown
2,750002,36,self-employed,married,primary,no,46,yes,yes,cellular,13,may,111,2,-1,0,unknown
3,750003,58,blue-collar,married,secondary,no,-1380,yes,yes,unknown,29,may,125,1,-1,0,unknown
4,750004,28,technician,single,secondary,no,1950,yes,no,cellular,22,jul,181,1,-1,0,unknown


In [7]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 750000 entries, 0 to 749999
Data columns (total 18 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   id         750000 non-null  int64 
 1   age        750000 non-null  int64 
 2   job        750000 non-null  object
 3   marital    750000 non-null  object
 4   education  750000 non-null  object
 5   default    750000 non-null  object
 6   balance    750000 non-null  int64 
 7   housing    750000 non-null  object
 8   loan       750000 non-null  object
 9   contact    750000 non-null  object
 10  day        750000 non-null  int64 
 11  month      750000 non-null  object
 12  duration   750000 non-null  int64 
 13  campaign   750000 non-null  int64 
 14  pdays      750000 non-null  int64 
 15  previous   750000 non-null  int64 
 16  poutcome   750000 non-null  object
 17  y          750000 non-null  int64 
dtypes: int64(9), object(9)
memory usage: 103.0+ MB


# EDA


# train model

In [8]:
from  sklearn.model_selection  import  train_test_split 
from  sklearn.preprocessing  import  OneHotEncoder 
from  sklearn.compose  import  ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier 
from sklearn.metrics import  classification_report, accuracy_score

In [9]:
X =  train_df.drop(columns=["id", "y"]) 
y = train_df["y"]

In [10]:
# ระบุคอลัมน์ categorical และ numeric
cat_cols = X.select_dtypes(include=["object"]).columns
num_cols = X.select_dtypes(exclude=["object"]).columns

In [11]:
# เก็บ id ของ test
test_ids = test_df["id"]

In [12]:
# Preprocessing: One-hot encoding + pass numeric
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols)
    ]
)

# Model pipeline
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=400, random_state=42))
])


In [13]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(600000, 16) (150000, 16) (600000,) (150000,)


In [14]:
X_train.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome
453635,28,blue-collar,single,secondary,no,5090,yes,yes,unknown,12,may,1297,2,-1,0,unknown
11651,51,technician,married,tertiary,no,1295,no,no,cellular,27,aug,119,9,-1,0,unknown
431999,57,management,divorced,tertiary,no,0,no,no,cellular,29,jan,87,1,-1,0,unknown
529211,48,blue-collar,single,primary,no,1323,yes,no,unknown,15,may,83,5,-1,0,unknown
110925,38,admin.,married,secondary,no,659,yes,no,cellular,28,jul,534,4,-1,0,unknown


In [ ]:
# Train
model.fit(X_train, y_train)

In [ ]:
# Predict
y_pred = model.predict(X_test)

In [ ]:
# Evaluate
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.95      0.97      0.96    131795
           1       0.76      0.65      0.70     18205

    accuracy                           0.93    150000
   macro avg       0.85      0.81      0.83    150000
weighted avg       0.93      0.93      0.93    150000



In [ ]:
# Predict
y_pred = model.predict(test_df)

# สร้าง submission DataFrame
submission = pd.DataFrame({
    "id": test_ids,
    "y": y_pred
})

# บันทึกเป็น CSV
submission.to_csv("submission.csv", index=False)

In [ ]:
!kaggle competitions submit -c playground-series-s5e8 -f submission.csv -m "Message"

100%|███████████████████████████████████████| 2.15M/2.15M [00:02<00:00, 904kB/s]
Successfully submitted to Binary Classification with a Bank Dataset